# 4. Interpretabilite du Modele

Visualiser ce que le modele voit pour chaque prediction via des saliency maps.
Aucun entrainement necessaire.

In [ ]:
import os, gc
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import matplotlib.cm as cm

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_logical_device_configuration(gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=4096)])
    except RuntimeError:
        for g in gpus:
            tf.config.experimental.set_memory_growth(g, True)

IMG_SIZE = 224
SEED = 42
np.random.seed(SEED)
PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'training' else os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'UTKFace')
OUTPUT_DIR = os.path.join(os.getcwd() if os.path.basename(os.getcwd()) == 'training' else os.path.join(os.getcwd(), 'training'), 'output_v2')
ETHNICITY_LABELS = ['White', 'Black', 'Asian', 'Indian', 'Other']
GENDER_LABELS = ['Male', 'Female']
print('Setup OK')

## 4.1 Classes + chargement images

In [ ]:
class MoEHead(layers.Layer):
    def __init__(self, num_experts, expert_dim, output_dim, name_prefix, **kwargs):
        super().__init__(**kwargs)
        self.num_experts = num_experts
        self.expert_dim = expert_dim
        self.output_dim = output_dim
        self.name_prefix = name_prefix
    def build(self, input_shape):
        self.expert_dense1 = [layers.Dense(self.expert_dim, activation='relu') for _ in range(self.num_experts)]
        self.expert_dropout = [layers.Dropout(0.2) for _ in range(self.num_experts)]
        self.expert_output = [layers.Dense(self.output_dim) for _ in range(self.num_experts)]
        self.gate_dense = layers.Dense(128, activation='relu')
        self.gate_output = layers.Dense(self.num_experts, activation='softmax')
        super().build(input_shape)
    def call(self, x, training=None):
        outs = [self.expert_output[i](self.expert_dropout[i](self.expert_dense1[i](x), training=training)) for i in range(self.num_experts)]
        stack = tf.stack(outs, axis=1)
        g = self.gate_output(self.gate_dense(x))
        return tf.reduce_sum(stack * tf.expand_dims(g, -1), axis=1)
    def get_config(self):
        c = super().get_config()
        c.update({'num_experts': self.num_experts, 'expert_dim': self.expert_dim, 'output_dim': self.output_dim, 'name_prefix': self.name_prefix})
        return c

class SparseFocalLoss(keras.losses.Loss):
    def __init__(self, gamma=2.0, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.class_weights = class_weights
    def call(self, y_true, y_pred):
        y_true = tf.reshape(tf.cast(y_true, tf.int32), [-1])
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        idx = tf.stack([tf.range(tf.shape(y_true)[0]), y_true], axis=1)
        p = tf.gather_nd(y_pred, idx)
        return tf.pow(1 - p, self.gamma) * (-tf.math.log(p))
    def get_config(self):
        return {**super().get_config(), 'gamma': self.gamma, 'class_weights': None}

CUSTOM = {'MoEHead': MoEHead, 'SparseFocalLoss': SparseFocalLoss}
print('Classes pretes')

In [ ]:
import random
random.seed(SEED)
files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('.jpg')])
picks = {}
for f in files:
    p = f.split('_')
    if len(p) < 4: continue
    try: age, g, e = int(p[0]), int(p[1]), int(p[2])
    except: continue
    if age < 5 and 'child' not in picks: picks['child'] = f
    elif 20 <= age <= 30 and g == 0 and 'young_m' not in picks: picks['young_m'] = f
    elif 20 <= age <= 30 and g == 1 and 'young_f' not in picks: picks['young_f'] = f
    elif 25 <= age <= 40 and e == 2 and 'asian' not in picks: picks['asian'] = f
    elif 25 <= age <= 40 and e == 1 and 'black' not in picks: picks['black'] = f
    elif age > 65 and e == 0 and 'old' not in picks: picks['old'] = f
    elif 20 <= age <= 35 and e == 3 and 'indian' not in picks: picks['indian'] = f
    elif age > 60 and g == 1 and 'old_f' not in picks: picks['old_f'] = f
    if len(picks) >= 8: break

sample_images, sample_labels = [], []
for f in picks.values():
    p = f.split('_')
    age, g, e = int(p[0]), int(p[1]), int(p[2])
    img = tf.io.read_file(os.path.join(DATA_DIR, f))
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    sample_images.append(img.numpy())
    sample_labels.append({'age': age, 'gender': GENDER_LABELS[g], 'eth': ETHNICITY_LABELS[e]})
sample_images = np.array(sample_images, dtype=np.float32)
print(f'{len(sample_images)} images chargees')

## 4.2 Fonction saliency map

In [ ]:
def saliency_map(model, img_batch, output_idx=None):
    inp = tf.Variable(tf.cast(img_batch, tf.float32))
    with tf.GradientTape() as tape:
        preds = model(inp, training=False)
        if isinstance(preds, list):
            pred = preds[output_idx]
        else:
            pred = preds
        if pred.shape[-1] > 1:
            target = tf.reduce_max(pred[0])
        else:
            target = pred[0, 0]
    grads = tape.gradient(target, inp)
    if grads is None:
        return np.zeros((IMG_SIZE, IMG_SIZE))
    sal = tf.reduce_max(tf.abs(grads[0]), axis=-1)
    sal = tf.expand_dims(tf.expand_dims(sal, 0), -1)
    sal = tf.nn.avg_pool2d(sal, ksize=15, strides=1, padding='SAME')
    sal = sal[0, :, :, 0]
    sal = sal / (tf.reduce_max(sal) + 1e-8)
    return sal.numpy()

def make_overlay(img, heatmap, alpha=0.45):
    h = tf.image.resize(heatmap[..., np.newaxis], (img.shape[0], img.shape[1])).numpy()[:,:,0]
    colored = cm.jet(h)[:,:,:3]
    return np.clip((1-alpha) * img/255.0 + alpha * colored, 0, 1)

print('Saliency map pret')

## 4.3 Visualisation — Modele Multitache

In [ ]:
best = os.path.join(OUTPUT_DIR, 'multitask_best.keras')
if not os.path.exists(best):
    best = os.path.join(OUTPUT_DIR, 'multitask_ft_a.keras')
if not os.path.exists(best):
    best = os.path.join(OUTPUT_DIR, 'multitask_warmup.keras')
model = keras.models.load_model(best, custom_objects=CUSTOM)
print(f'Modele charge: {model.name}')

n = min(6, len(sample_images))
fig, axes = plt.subplots(n, 4, figsize=(18, 4.5*n))
for i in range(n):
    batch = np.expand_dims(sample_images[i], 0)
    pa, pg, pe = model.predict(batch, verbose=0)
    p_age = int(pa[0][0])
    p_gen = GENDER_LABELS[1 if pg[0][0] > 0.5 else 0]
    p_eth = ETHNICITY_LABELS[np.argmax(pe[0])]
    axes[i,0].imshow(sample_images[i].astype(np.uint8))
    axes[i,0].set_title(f'Vrai: {sample_labels[i]["age"]}a {sample_labels[i]["gender"]} {sample_labels[i]["eth"]}\nPred: {p_age}a {p_gen} {p_eth}', fontsize=9, fontweight='bold')
    axes[i,0].axis('off')
    for j, (idx, name) in enumerate([(0,'AGE'), (1,'GENRE'), (2,'ETHNICITE')]):
        hm = saliency_map(model, batch, output_idx=idx)
        axes[i,j+1].imshow(make_overlay(sample_images[i], hm))
        axes[i,j+1].set_title(name, fontsize=10, fontweight='bold')
        axes[i,j+1].axis('off')
plt.suptitle('Saliency Maps - Multitache MoE (EfficientNetB0)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'gradcam_multitache.png'), dpi=150, bbox_inches='tight')
plt.show()
del model
keras.backend.clear_session()
gc.collect()
print('Multitache OK')

## 4.4 Visualisation — Modeles Individuels

In [ ]:
n = min(6, len(sample_images))
configs = [('Age', os.path.join(OUTPUT_DIR, 'age_best.keras')),
           ('Genre', os.path.join(OUTPUT_DIR, 'gender_best.keras')),
           ('Ethnicite', os.path.join(OUTPUT_DIR, 'ethnicity_best.keras'))]
all_hm, all_pr = {}, {}
for name, path in configs:
    keras.backend.clear_session()
    gc.collect()
    if not os.path.exists(path):
        print(f'{name}: non trouve'); all_hm[name] = [None]*n; all_pr[name] = [None]*n; continue
    m = keras.models.load_model(path, custom_objects=CUSTOM)
    print(f'{name} charge')
    hms, prs = [], []
    for i in range(n):
        batch = np.expand_dims(sample_images[i], 0)
        pr = m.predict(batch, verbose=0)
        prs.append(pr)
        hms.append(saliency_map(m, batch))
    all_hm[name] = hms; all_pr[name] = prs
    del m; keras.backend.clear_session(); gc.collect()
    print(f'{name} OK')

fig, axes = plt.subplots(n, 4, figsize=(18, 4.5*n))
for i in range(n):
    axes[i,0].imshow(sample_images[i].astype(np.uint8))
    axes[i,0].set_title(f'Vrai: {sample_labels[i]["age"]}a {sample_labels[i]["gender"]} {sample_labels[i]["eth"]}', fontsize=9, fontweight='bold')
    axes[i,0].axis('off')
    for j, (name, _) in enumerate(configs):
        hm = all_hm[name][i]
        pr = all_pr[name][i]
        if hm is not None:
            axes[i,j+1].imshow(make_overlay(sample_images[i], hm))
            if name == 'Age': val = f'{int(pr[0][0])} ans'
            elif name == 'Genre': val = GENDER_LABELS[1 if pr[0][0] > 0.5 else 0]
            else: val = ETHNICITY_LABELS[np.argmax(pr[0])]
            axes[i,j+1].set_title(f'{name}: {val}', fontsize=10, fontweight='bold')
        else:
            axes[i,j+1].imshow(sample_images[i].astype(np.uint8))
            axes[i,j+1].set_title(f'{name}: N/A', fontsize=10)
        axes[i,j+1].axis('off')
plt.suptitle('Saliency Maps - 3 Modeles Individuels (EfficientNetB0)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'gradcam_individuels.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Individuels OK')

## 4.5 Comment le modele fait-il ses predictions ?

Le modele analyse uniquement les pixels de l'image. Il n'utilise ni infrarouge ni capteur special.

**Age** : rides, texture de peau, cheveux gris, rondeur du visage (enfants)

**Genre** : machoire, pilosite faciale, structure osseuse frontale

**Ethnicite** : teint, forme des yeux et du nez, proportions du visage

Le modele peut etre biaise par le desequilibre du dataset (plus de White que d'Other).